# Rerun Weng et al. (2024): STGAT

**Paper:** Jiaqi Weng and colleagues, *Graph Representation Learning for Dengue Forecasting*, IEEE BigData 2024. The full author details are in the project's reference records.

**Code:** [the authors' repository](https://github.com/MLOpenSourceOpenScience/disease_modeling_MLOS2), commit `45f1c08`.

This notebook runs the authors' `run_stgat()` function with data prefixes `[0.6, 0.7, 0.8, 0.9, 1.0]`. A prefix is the first part of the time series. We use their model, training loop, data handling and scoring code rather than replacing those parts with our own implementation.

## Original target

The targets below come from the authors' `Models/results.txt` and printed table. They are not this notebook's achieved scores.

| Source | MAE | RMSE |

|---|---|---|
| `results.txt` (raw) | 25.38192804812382 ± 1.37234628300591 | 44.78221112357350 ± 2.26308189406910 |
| Table I (as printed) | 25.38 ± 1.37 | 44.78 ± 2.26 |

## Recorded software changes

The Kaggle host used Python 3.12 in the earlier run. The notebook creates a separate Python 3.11 environment so it can install the authors' `torch==2.1.2`.

It installs `torch_geometric_temporal==0.54.0` with `--no-deps` because its declared `pandas<=1.3.5` conflicts with the authors' pandas 2.2 setting. It also uses `torch_geometric==2.4.0` instead of 2.5.3 so the older temporal package can import the required module.

The other named versions stay `numpy~=1.26.2`, `pandas~=2.2.0`, `scikit_learn==1.4.0` and `statsmodels==0.14.1`, with the original torch version above. Report these environment changes when discussing the rerun. Read `references/REPRODUCTION_SCOPE.md` in the public repository for achieved agreement and limitations.


## Understand the authors' scoring before reading the result

The released array has shape `(459 weeks, 25 districts, 11 features)`. The case channel is index 5, also written `x[..., -6]`. Their GNN runner uses only that case channel, not the ten weather channels.

The original code calculates a mean and standard deviation from the whole series before selecting a data prefix. This includes future test-period values. It makes three-week inputs and three-week targets. Its binary 25-by-25 district graph includes self-links and has 141 directed edges. District order follows sorted JSON keys.

For each prefix, a model trains on the first 70% for 50 epochs, with Adam, learning rate `1e-4`, weight decay `5e-5`, MSE loss and batch size 1. It uses the final epoch rather than validation-based early stopping. The code sets Python, NumPy and PyTorch seeds to zero; hardware and software can still cause numerical differences.

The reported RMSE is an average of per-window RMSEs. This differs from calculating one RMSE over all prediction errors together. The code also computes a held-out test score but records a score from the full prefix, including training windows, for its reported cross-validation calculation. It reports the mean and population standard deviation over five prefixes.

This notebook preserves those choices to examine the released experiment. Reproducing its numbers does not make its evaluation protocol a matched comparison with LEG-SEIR's 2026 study.


## 1. Set up the recorded package versions in Python 3.11


In [ ]:
import subprocess, sys, os, json, time, hashlib, re
from pathlib import Path

REPO = "https://github.com/MLOpenSourceOpenScience/disease_modeling_MLOS2.git"
COMMIT = "45f1c0878f002407633ed1237638734faa9ceb2b"
BLOB_NPY = "f7cfa6ec31a4058584fe256a1d6de6800e72a5b1"
BLOB_ADJ = "f3a3cb7f43998850410b0a494f16f331c3830a84"
SEGMENTS = [0.6, 0.7, 0.8, 0.9, 1.0]     # the authors' __main__ segment list

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
# Build outside /kaggle/working: anything left there becomes kernel output, and a
# 40 MB clone plus a venv makes `kaggle kernels output` unusably slow.
SCRATCH = Path("/tmp/repro") if Path("/tmp").exists() else WORK
SCRATCH.mkdir(parents=True, exist_ok=True)
SRC = SCRATCH / "mlos2"
VENV = SCRATCH / "venv311"
PY311 = VENV / "bin" / "python"

os.environ["MPLBACKEND"] = "Agg"          # their plotting helpers call plt.show()


def sh(*args, check=True, quiet=False, **kw):
    """Run a command, echoing it, and abort on a non-zero exit."""
    if not quiet:
        print("$", " ".join(str(a) for a in args))
    r = subprocess.run([str(a) for a in args], text=True, capture_output=True, **kw)
    if r.stdout.strip() and not quiet:
        print(r.stdout[-2000:])
    if r.returncode != 0:
        print(r.stderr[-4000:])
        if check:
            raise SystemExit("command failed: " + " ".join(str(a) for a in args))
    return r


print("kernel python:", sys.version.split()[0])

In [ ]:
# Kaggle runs Python 3.12; torch 2.1.2 has no cp312 wheel. Fetch a standalone
# 3.11 with uv rather than bumping the authors' pinned torch.
sh(sys.executable, "-m", "pip", "install", "-q", "uv")

UV = [sys.executable, "-m", "uv"]
sh(*UV, "python", "install", "3.11")
sh(*UV, "venv", "--python", "3.11", str(VENV))

PIP = [*UV, "pip", "install", "-q", "--python", str(PY311)]

# Exactly the versions in the authors' requirements.txt.
sh(*PIP, "torch==2.1.2", "--index-url", "https://download.pytorch.org/whl/cpu")
sh(*PIP, "torch_scatter", "torch_sparse", "-f",
   "https://data.pyg.org/whl/torch-2.1.2+cpu.html")

# Deviation 2: the authors pin torch_geometric==2.5.3, but PGT 0.54.0 imports
# torch_geometric.utils.to_dense_adj, which PyG removed in 2.4. 2.4.0 is the
# newest release where all five architectures import.
sh(*PIP, "torch_geometric==2.4.0", "numpy~=1.26.2", "pandas~=2.2.0",
   "scikit_learn==1.4.0", "statsmodels==0.14.1", "decorator==4.4.2",
   "cython", "matplotlib", "tqdm")

# Deviation 1: PGT's own pandas<=1.3.5 pin contradicts the authors'
# pandas~=2.2.0 and has no Python 3.11 wheel.
sh(*PIP, "--no-deps", "torch_geometric_temporal==0.54.0")

In [ ]:
probe = sh(str(PY311), "-c", """
import json, torch, torch_geometric, pandas, numpy
from torch_geometric_temporal import A3TGCN, ASTGCN, AAGCN
from torch_geometric_temporal.nn.recurrent import DCRNN
from torch_geometric_temporal.signal import StaticGraphTemporalSignal, temporal_signal_split
print(json.dumps({
    "python": ".".join(map(str, __import__("sys").version_info[:3])),
    "torch": torch.__version__,
    "torch_geometric": torch_geometric.__version__,
    "pandas": pandas.__version__,
    "numpy": numpy.__version__,
}))
""", quiet=True)

versions = json.loads(probe.stdout.strip().splitlines()[-1])
print(json.dumps(versions, indent=2))
assert versions["python"].startswith("3.11"), versions["python"]
assert versions["torch"].startswith("2.1.2"), versions["torch"]
assert versions["torch_geometric"] == "2.4.0", versions["torch_geometric"]
print("all five architectures import OK under Python 3.11")

## 2. Download the authors' code and check file fingerprints


In [ ]:
if not SRC.exists():
    sh("git", "clone", REPO, str(SRC))
sh("git", "-C", str(SRC), "checkout", "--quiet", COMMIT)

head = sh("git", "-C", str(SRC), "rev-parse", "HEAD", quiet=True).stdout.strip()
assert head == COMMIT, f"expected {COMMIT}, got {head}"
print("repository at", head)

# Verify the released inputs by git blob SHA. This is platform-independent --
# unlike an md5 of the checked-out bytes, which differs between a Windows and a
# Linux clone because git rewrites line endings in text files.
for rel, want in [("Data/Datasets/sri_lanka_2013-2022_shifted.npy", BLOB_NPY),
                  ("Models/sri_lanka_adj_list.json", BLOB_ADJ)]:
    got = sh("git", "-C", str(SRC), "rev-parse", f"HEAD:{rel}", quiet=True).stdout.strip()
    print(("OK  " if got == want else "MISMATCH ") + f"{rel}  {got}")
    assert got == want, f"{rel} blob changed: {got} != {want}"

## 3. Run the authors' training code

The imported evaluation module sets random seeds to zero. The next cell trains on five prefixes for 50 epochs each on CPU. Earlier runs took about 1-4 hours, depending on the architecture and machine. The code prints its training settings.


In [ ]:
RUNNER = "run_stgat"
MODEL = "STGAT"

script = (
    "import sys\n"
    "sys.path.insert(0, '.')\n"
    "import evaluation\n"
    "print(\"authors' hyperparameters, read from their module:\")\n"
    "for _n in ('batch_size','window_size','in_channels','out_channels',"
    "'num_nodes','num_epochs','lr','decay','dropout','data_file'):\n"
    "    print('  %-12s = %s' % (_n, getattr(evaluation, _n)))\n"
    f"evaluation.run_stgat({SEGMENTS})\n"
)

started = time.time()
proc = subprocess.Popen(
    [str(PY311), "-u", "-c", script], cwd=str(SRC / "Models"),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
lines = []
for line in proc.stdout:            # stream, so progress shows in the Kaggle log
    lines.append(line)
    if any(k in line for k in ("Average", "= ", "Test,", "Full,")):
        print(line, end="")
proc.wait()
elapsed = time.time() - started
captured = "".join(lines)

(WORK / (MODEL + "_stdout.txt")).write_text(captured, encoding="utf-8")
print(f"\n{RUNNER} exited {proc.returncode} after {elapsed / 60:.1f} min")
if proc.returncode != 0:
    print(captured[-3000:])
    raise SystemExit("the authors' runner failed")

## 4. Compare the new output with the original targets


In [ ]:
TARGET = dict(mae=25.38192804812382, mae_sd=1.3723462830059112, rmse=44.7822111235735, rmse_sd=2.2630818940691015)
PAPER = dict(mae=25.38, rmse=44.78)
print(TARGET)
print(PAPER)

In [ ]:
# Their run_* functions end with two lines:
#   Average MAE: <x>, Standard Deviation: <y>
#   Average RMSE: <x>, Standard Deviation: <y>
def parse(text, metric):
    hits = re.findall(
        r"Average " + metric + r": ([\d.eE+-]+), Standard Deviation: ([\d.eE+-]+)", text
    )
    if not hits:
        raise ValueError("could not find 'Average " + metric + "' in the captured output")
    return float(hits[-1][0]), float(hits[-1][1])


got_mae, got_mae_sd = parse(captured, "MAE")
got_rmse, got_rmse_sd = parse(captured, "RMSE")

print("metric      this run          results.txt         abs diff     rel %    Table I")
for name, got, target, paper in [
    ("MAE", got_mae, TARGET["mae"], PAPER["mae"]),
    ("RMSE", got_rmse, TARGET["rmse"], PAPER["rmse"]),
]:
    diff = got - target
    print(f"{name:6s} {got:18.12f} {target:18.12f} {diff:12.2e} "
          f"{100 * diff / target:8.4f} {paper:9.2f}")

print()
print("sd     this run          results.txt")
print(f"{'MAE':6s} {got_mae_sd:18.12f} {TARGET['mae_sd']:18.12f}")
print(f"{'RMSE':6s} {got_rmse_sd:18.12f} {TARGET['rmse_sd']:18.12f}")

In [ ]:
import csv

row = {
    "model": MODEL, "runner": RUNNER, "segments": json.dumps(SEGMENTS),
    "commit": COMMIT, **{f"env_{k}": v for k, v in versions.items()},
    "minutes": round(elapsed / 60, 2),
    "mae": got_mae, "mae_sd": got_mae_sd, "rmse": got_rmse, "rmse_sd": got_rmse_sd,
    "target_mae": TARGET["mae"], "target_mae_sd": TARGET["mae_sd"],
    "target_rmse": TARGET["rmse"], "target_rmse_sd": TARGET["rmse_sd"],
    "paper_mae": PAPER["mae"], "paper_rmse": PAPER["rmse"],
    "mae_rel_error_pct": 100 * (got_mae - TARGET["mae"]) / TARGET["mae"],
    "rmse_rel_error_pct": 100 * (got_rmse - TARGET["rmse"]) / TARGET["rmse"],
}
out = WORK / ("weng2024_" + MODEL + "_reproduction.csv")
with open(out, "w", newline="", encoding="utf-8") as fh:
    writer = csv.DictWriter(fh, fieldnames=list(row))
    writer.writeheader()
    writer.writerow(row)
print("wrote", out)
for k, v in row.items():
    print(f"  {k:22s} {v}")

## 5. Interpret the comparison

Look at the actual MAE/RMSE difference, package versions, checksums and scoring procedure. A small percentage difference is not proof that two runs produced identical predictions. The historical workflow used 0.01% and 5% as practical comparison thresholds, not statistical tests or universal definitions of reproduction.

Five GNN rows on the shifted dataset were rerun. Missing scripts or data prevented a complete rerun of the original 20 rows. See the project scope notes and saved CSVs for details. These results are not matched competitors in the LEG-SEIR 2026 results table.

The original code's whole-series scaling, per-window RMSE averaging and use of training windows in its reported comparison limit what the reproduced numbers show. Keep those limits visible when reporting them.
